# G04 — Unified six-source ensemble benchmark (seed 42)

**Purpose:** Apply the *same six base-model types* and *identical, prespecified ensemble formulas* to both original image-level and N01 lesion-aware predictions. Score ISIC 2019 **HAM+BCN source-matched pooled subsets**, HAM10000 source subset and BCN20000 source subset, plus an optional full N01 validation replay. No image downloading, network, training, source-specific fitting or class-bias tuning.

**Kaggle notebook inputs (attach OUTPUTS, not .ipynb code):**
- `g03a-image-level-all-ensembles-s42` containing `reports/final_status.json` and `predictions/ensemble_probabilities.npz`.
- `g03b-lesion-aware-all-ensembles-s42` with the same two artifact types.
- Optional: `lesion-r07-n01-all-ensembles-s42` output for a separate **complete 2,186-image N01 validation** check, using six frozen raw base models.
- Optional: `skin-lesion-complete-per-image-results-no-xai` output containing `01_PER_IMAGE_RESULTS/ISIC2019_test/probabilities.npz` for the original **complete 2,534-image J1 test**; not part of the source-based selection.

**Interpretation:** Original J1 test and N01 validation were previously used in experimentation. Selecting a winner from this comparison is *exploratory and selection-biased*, NOT a final independent publication estimate. The two tracks use different evaluation subsets. HAM/BCN here are *ISIC 2019 source subsets*, not independent held-out target-source cohorts. `ISIC2019_pooled_source_matched` is **not** the full 2,534 J1 test / 2,186 N01 validation.

**Fairness constraint:** No supervised, DINOv2, TTA-only, trained stacker or learned class biases in the shared catalogue, because they are not identically available in both six-source inference archives. These are retained as historical comparison results elsewhere, not falsely represented as the same ensemble.

**v2 compatibility fix:** G03-A/G03-B saved image identifiers and group/source fields as NumPy object-string arrays. These trusted, previously generated Kaggle archives require `allow_pickle=True` for loading. All text fields are then type-checked; probabilities and labels are checked for numeric dtypes. **Attach only your own trusted G03 outputs.**

In [1]:
import os,json,hashlib,time,math
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score,log_loss

SEED=42
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
SOURCES=['Rotation','SimCLR','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
INPUT=Path(os.getenv('G04_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').is_dir() else '/mnt/data'))
WORK=Path(os.getenv('G04_WORK_DIR','/kaggle/working' if Path('/kaggle/working').is_dir() else '/mnt/data'))
OUT=WORK/'g04-unified-six-source-ensembles-s42'
for name in ['tables','reports','predictions']:(OUT/name).mkdir(parents=True,exist_ok=True)
START=time.monotonic();MAX_HOURS=8.5

def req(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
def save_json(path,data):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_suffix(p.suffix+'.tmp')
    t.write_text(json.dumps(data,indent=2,allow_nan=False,default=str));t.replace(p)
def norm(p):
    x=np.asarray(p,dtype=np.float64)
    req(x.ndim==2 and x.shape[1]==8 and np.isfinite(x).all() and (x>=0).all(),'Invalid [N,8] model probability matrix')
    req(np.max(np.abs(x.sum(1)-1))<2e-3,'Model probabilities must sum to 1')
    x=np.clip(x,1e-12,None);return x/x.sum(1,keepdims=True)
def canonical_s(s):return str(s).strip().upper()

def archive_for(stage):
    matches=[]
    for p in INPUT.rglob('final_status.json'):
        try:
            s=json.loads(p.read_text())
            if s.get('stage')==stage and s.get('status')=='COMPLETED':
                npz=p.parent.parent/'predictions/ensemble_probabilities.npz'
                if npz.is_file():matches.append((p,npz,s))
        except (OSError,ValueError,KeyError):continue
    req(len(matches)==1,f'Attach exactly one completed {stage} Kaggle output with `predictions/ensemble_probabilities.npz`. Found {len(matches)}: {[str(x[0]) for x in matches[:4]]}')
    return matches[0]

BASE_KEYS={
 'G03-A':{'Rotation':'Base_Rotation','SimCLR':'Base_SimCLR_NoTTA',
          'FusionConcat':'Base_FusionConcat','FusionGatedV2':'Base_FusionGatedV2',
          'FusionGatedV3':'Base_FusionGatedV3','ResidualFusion':'Base_ResidualFusion'},
 'G03-B':{'Rotation':'Base_rotation','SimCLR':'Base_simclr',
          'FusionConcat':'Base_FusionConcat','FusionGatedV2':'Base_FusionGatedV2',
          'FusionGatedV3':'Base_FusionGatedV3','ResidualFusion':'Base_ResidualFusion'}
}


In [2]:

def npz_text(z,key):
    a=z[key]
    req(a.ndim==1 and a.dtype.kind in 'USO',f'{key}: expected one-dimensional text array')
    if a.dtype.kind=='O':
        req(all(isinstance(v,(str,np.str_)) for v in a),f'{key}: non-text Python objects found')
    return a.astype(str)

def load_g03(stage):
    st_path,path,status=archive_for(stage)
    with np.load(path,allow_pickle=True) as z:
        req(set(['image_id','y','source','group_id','classes','candidate_names','proba']).issubset(z.files),f'{stage} incomplete NPZ')
        req(npz_text(z,'classes').tolist()==CLASSES,f'{stage} class-order mismatch')
        names=npz_text(z,'candidate_names').tolist()
        req(len(names)==len(set(names))==int(status['candidate_count']),f'{stage} candidate-count mismatch')
        ids=npz_text(z,'image_id');y=z['y'].astype(int)
        source=npz_text(z,'source');groups=npz_text(z,'group_id')
        prob=z['proba']
        req(prob.dtype.kind=='f' and z['y'].dtype.kind in 'iu','G03 probabilities or labels are not numeric arrays')
        req(prob.shape==(len(names),len(ids),8),f'{stage} malformed ensemble archive dimensions')
        req(len(ids)==int(status['n_images']) and len(set(ids))==len(ids),f'{stage} invalid image IDs')
        req(set(np.unique(source))=={'HAM10000','BCN20000'},f'{stage} source memberships unexpected')
        req(((y>=0)&(y<8)).all(),f'{stage} invalid class IDs')
        keys=BASE_KEYS[stage]
        req(set(keys.values()).issubset(names),f'{stage} missing required single-model sources: {sorted(set(keys.values())-set(names))}')
        index={n:i for i,n in enumerate(names)}
        raw={s:norm(prob[index[keys[s]]]) for s in SOURCES}
    meta=pd.DataFrame({'image_id':ids,'y':y,'source':source,'group_id':groups})
    # CSV SHA verification is also checked against the source G03 completion record.
    desc={'stage':stage,'status_file':str(st_path),'status_sha256':sha(st_path),
          'predictions_file':str(path),'predictions_sha256':sha(path),
          'n_images':len(meta),'per_source':meta.source.value_counts().to_dict(),
          'candidate_count_in_original_G03':int(status['candidate_count'])}
    return meta,raw,desc

meta_a,raw_a,prov_a=load_g03('G03-A')
meta_b,raw_b,prov_b=load_g03('G03-B')
TRACKS=OrderedDict([('image_aware',(meta_a,raw_a)),('lesion_aware',(meta_b,raw_b))])
print('Loaded original six-source probabilities:')
for x in (prov_a,prov_b):print(x['stage'],x['n_images'],x['per_source'])


Loaded original six-source probabilities:
G03-A 2239 {'BCN20000': 1240, 'HAM10000': 999}
G03-B 2109 {'HAM10000': 1060, 'BCN20000': 1049}


## Frozen, identical six-source candidate catalogue
All candidates below are untrained formulae, with the *same* component names, weights, temperature-free probabilities and beta values under both protocols. Fixed combinations and individual sources are not retroactively called “validated” by this benchmark.

In [3]:
RECIPES=OrderedDict()
def add(name,family,kind,sources,weights=None,beta=None,measure=None):
    req(name not in RECIPES,'Duplicate recipe ID')
    req(set(sources).issubset(SOURCES) and len(sources)==len(set(sources)),'Unknown or repeated model input')
    RECIPES[name]={'family':family,'kind':kind,'sources':sources,'weights':weights,'beta':beta,'measure':measure}
for s in SOURCES:add('Base_'+s,'Single baseline','single',[s])
subsets=OrderedDict([
    ('All6',SOURCES),('SSL2',SOURCES[:2]),('Fusion4',SOURCES[2:]),
    ('SimCLRResidual',['SimCLR','ResidualFusion']),
    ('SimCLRConcat',['SimCLR','FusionConcat']),
    ('RotationSimCLR',['Rotation','SimCLR'])
])
for sub,keys in subsets.items():
    for kind in ['arithmetic','geometric','vote']:
        add(f'{kind}_{sub}',f'Fixed {kind}',kind,keys)
for w in [.4,.6,.8]:
    add(f'Weighted_SimCLRResidual_{w:g}','Prespecified weighting','arithmetic',
        ['SimCLR','ResidualFusion'],[w,1-w])
    add(f'Weighted_SimCLRConcat_{w:g}','Prespecified weighting','arithmetic',
        ['SimCLR','FusionConcat'],[w,1-w])
add('BalancedBranches_SSL50_Fusion50','Two-branch equal weighting','grouped',SOURCES)
for sub in ['All6','SimCLRResidual']:
    for m in ['confidence','entropy','margin']:
        for beta in [2.,4.]:
            add(f'Gate_{m}_{sub}_beta{beta:g}','Prespecified adaptive gate','gate',
                subsets[sub],beta=beta,measure=m)

req(len(RECIPES)>25,'Candidate catalogue unexpectedly small')
save_json(OUT/'reports/unified_frozen_recipes.json',RECIPES)
print('Exactly identical frozen candidate formulas for both tracks:',len(RECIPES))
print('Families:',pd.Series([x['family'] for x in RECIPES.values()]).value_counts().to_dict())

def ensemble(raw,recipe):
    sources=recipe['sources'];kind=recipe['kind']
    if kind=='single':return norm(raw[sources[0]])
    if kind=='grouped':
        ssl=.5*(raw['Rotation']+raw['SimCLR'])
        fusion=.25*sum((raw[k] for k in SOURCES[2:]),start=np.zeros_like(ssl))
        return norm(.5*ssl+.5*fusion)
    p=np.stack([raw[s] for s in sources],axis=0)
    w=np.ones(len(sources),dtype=float)/len(sources) if recipe['weights'] is None else np.asarray(recipe['weights'],dtype=float)
    req(w.shape==(len(sources),) and (w>=0).all() and np.isclose(w.sum(),1),'Invalid candidate weights')
    if kind=='arithmetic':return norm(np.einsum('snc,s->nc',p,w,optimize=True))
    if kind=='geometric':return softmax(np.einsum('snc,s->nc',np.log(np.clip(p,1e-12,1)),w,optimize=True),axis=1)
    if kind=='vote':
        onehot=np.eye(8)[p.argmax(2)]
        return norm(np.einsum('snc,s->nc',onehot,w,optimize=True)+1e-4*np.einsum('snc,s->nc',p,w,optimize=True))
    if kind=='gate':
        q=np.transpose(p,(1,0,2))
        if recipe['measure']=='confidence':score=q.max(2)
        elif recipe['measure']=='entropy':score=1.+(q*np.log(np.clip(q,1e-12,1))).sum(2)/math.log(8)
        elif recipe['measure']=='margin':
            top=np.partition(q,-2,axis=2)[:,:,-2:];score=top.max(2)-top.min(2)
        else:raise RuntimeError('Unsupported gate')
        wt=softmax(float(recipe['beta'])*score,axis=1)
        return norm(np.einsum('nsc,ns->nc',q,wt,optimize=True))
    raise RuntimeError('Unsupported recipe kind')

# Local correctness tests, independent of actual Kaggle data
rng=np.random.default_rng(42);dummy={s:rng.dirichlet(np.ones(8),size=11) for s in SOURCES}
req(np.allclose(ensemble(dummy,RECIPES['arithmetic_All6']),sum(dummy.values())/6,atol=1e-12),'Mean formula validation failed')
req(np.allclose(ensemble(dummy,RECIPES['BalancedBranches_SSL50_Fusion50']),.25*(dummy['Rotation']+dummy['SimCLR'])+.125*sum((dummy[k] for k in SOURCES[2:]),start=np.zeros((11,8))),atol=1e-12),'Branch weighting formula validation failed')
for name,formula in RECIPES.items():
    p=ensemble(dummy,formula)
    req(p.shape==(11,8) and np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-9),f'Bad recipe {name}')
print('All',len(RECIPES),'candidate formulas passed invariant tests')


Exactly identical frozen candidate formulas for both tracks: 43
Families: {'Prespecified adaptive gate': 12, 'Fixed arithmetic': 6, 'Single baseline': 6, 'Fixed geometric': 6, 'Fixed vote': 6, 'Prespecified weighting': 6, 'Two-branch equal weighting': 1}
All 43 candidate formulas passed invariant tests


In [4]:
def ece(y,p,bins=15):
    a=p.argmax(1);conf=p.max(1);edges=np.linspace(0,1,bins+1);out=0.
    for k in range(bins):
        mask=(conf>=edges[k])&(conf<(edges[k+1] if k<bins-1 else 1.+1e-8))
        if mask.any():out+=float(mask.mean())*abs(float((a[mask]==y[mask]).mean())-float(conf[mask].mean()))
    return out

def metrics(y,p):
    y=np.asarray(y,dtype=int);p=norm(p);pred=p.argmax(1);present=np.unique(y)
    req(len(y)>0 and len(present)>=2,'Insufficient distinct ground-truth classes')
    rec=[float((pred[y==c]==c).mean()) for c in present]
    unsupported=float((~np.isin(pred,present)).mean())
    return {'n':len(y),'n_classes':len(present),'classes_present':';'.join(CLASSES[c] for c in present),
            'balanced_accuracy':float(np.mean(rec)),'accuracy':float((pred==y).mean()),
            'macro_f1':float(f1_score(y,pred,labels=present,average='macro',zero_division=0)),
            'macro_f1_eight':float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
            'nll_eight':float(log_loss(y,p,labels=np.arange(8))),
            'ece15':float(ece(y,p)),'unsupported_prediction_rate':unsupported}

ROWS=[];PER_CLASS=[];PREDICTIONS={}
for track,(meta,raw) in TRACKS.items():
    n=len(meta);y=meta.y.to_numpy(dtype=int);source=meta.source.to_numpy(dtype=str)
    preds=OrderedDict()
    for method,spec in RECIPES.items():
        req(time.monotonic()-START<MAX_HOURS*3600,'Runtime guard reached; outputs remain reproducible')
        p=ensemble(raw,spec);preds[method]=p.astype(np.float32)
        for cohort in ['ISIC2019_pooled_source_matched','HAM10000','BCN20000']:
            idx=np.arange(n) if cohort=='ISIC2019_pooled_source_matched' else np.flatnonzero(source==cohort)
            if len(idx)==0:continue
            yy=y[idx];pp=p[idx];stat=metrics(yy,pp)
            ROWS.append({'track':track,'cohort':cohort,'method':method,'family':spec['family'],**stat})
            pr=pp.argmax(1)
            for c in np.unique(yy):
                m=yy==c
                PER_CLASS.append({'track':track,'cohort':cohort,'method':method,'class':CLASSES[c],
                    'support':int(m.sum()),'recall':float((pr[m]==c).mean())})
    PREDICTIONS[track]=preds
    np.savez_compressed(OUT/'predictions'/f'{track}_unified_probabilities.npz',
        image_id=meta.image_id.astype(str).to_numpy(),group_id=meta.group_id.astype(str).to_numpy(),
        source=source,y=y.astype(np.int32),classes=np.asarray(CLASSES),
        methods=np.asarray(list(preds)),proba=np.stack(list(preds.values()),axis=0))
board=pd.DataFrame(ROWS);recalls=pd.DataFrame(PER_CLASS)
board.to_csv(OUT/'tables/all_unified_candidate_metrics.csv',index=False)
recalls.to_csv(OUT/'tables/per_class_recall_all_candidates.csv',index=False)
req(board.groupby('track').method.nunique().eq(len(RECIPES)).all(),'Different ensemble catalogues between protocols')
print('Unified six-source comparison:',len(RECIPES),'candidate definitions,',len(board),'metric rows')


Unified six-source comparison: 43 candidate definitions, 258 metric rows


## Cross-protocol ranking and provisional choice

**Primary comparison = four source-stratified cells:** image-aware HAM, image-aware BCN, lesion-aware HAM, lesion-aware BCN. They receive equal weighting. The pooled ISIC source-matched score is displayed separately; adding it to the average would double-count the same images. The selection rule below is fixed before looking at the scores: maximize mean four-cell BACC, break ties by higher worst-cell BACC, then mean macro-F1 and method name. This remains **exploratory**, because the J1 test has already been evaluated and the R07 N01 validation was previously used for ensemble selection.

In [5]:
four=board[board.cohort.isin(['HAM10000','BCN20000'])].copy()
four['cell']=four.track+'__'+four.cohort
req(four.cell.nunique()==4 and four.groupby('method').cell.nunique().eq(4).all(),'Missing one or more of the four protocol/source cells')
piv=four.pivot(index='method',columns='cell',values='balanced_accuracy')
cols=['image_aware__HAM10000','image_aware__BCN20000','lesion_aware__HAM10000','lesion_aware__BCN20000']
piv=piv[cols]
score=piv.copy();score['mean_four_source_bacc']=piv.mean(1)
score['worst_source_bacc']=piv.min(1)
score['spread_bacc']=piv.max(1)-piv.min(1)
f1=four.groupby('method').macro_f1.mean()
score['mean_four_source_macro_f1']=f1
ref=piv.loc['arithmetic_All6']
for c in cols:score['gain_vs_meanall6__'+c]=score[c]-ref[c]
score['n_cells_better_than_meanall6']=(piv.gt(ref,axis=1)).sum(1)
score['n_cells_no_worse_than_meanall6']=(piv.ge(ref-1e-12,axis=1)).sum(1)
score['family']=[RECIPES[m]['family'] for m in score.index]
score=score.reset_index().rename(columns={'index':'method'})
score=score.sort_values(['mean_four_source_bacc','worst_source_bacc','mean_four_source_macro_f1','method'],ascending=[False,False,False,True]).reset_index(drop=True)
score.insert(0,'exploratory_rank',np.arange(1,len(score)+1))
score.to_csv(OUT/'tables/unified_4cell_exploratory_rankings.csv',index=False)
for c in cols:
    names=score.sort_values(c,ascending=False).method.head(10).tolist()
    print('\n',c,'top 10:')
    print((100*score.set_index('method').loc[names,[c]]).round(2).to_string())
print('\nOVERALL TOP 20 (percent):')
main_cols=['exploratory_rank','method','mean_four_source_bacc','worst_source_bacc']+cols
show=score[main_cols].head(20).copy()
for c in main_cols[2:]:show[c]=100*show[c]
print(show.round(2).to_string(index=False))

primary=score.loc[~score.method.str.startswith('Base_')].iloc[0]
provisional=str(primary.method)
print('\nProvisional ensemble-only leader:',provisional)
print('Provisional ensemble wins all four cells?',bool(all(float(primary[c])>=score[c].max()-1e-12 for c in cols)))

# No parameter fitting or post-hoc method calibration is performed. This file is an
# EXPERIMENTAL RECIPE for later independent confirmation, not a validated final model.
save_json(OUT/'reports/provisional_ensemble_recipe_NOT_FINAL.json',{
    'status':'EXPLORATORY_NOT_INDEPENDENTLY_VALIDATED','selection':'highest mean four-cell BACC; tie-break min BACC, mean macro-F1, name',
    'method':provisional,'recipe':RECIPES[provisional],'classes':CLASSES,'sources':SOURCES,
    'four_cell_bacc':{k:float(primary[k]) for k in cols},
    'mean_four_cell_bacc':float(primary.mean_four_source_bacc),
    'selection_bias_note':'J1 test already examined; R07 previously selected ensembles using N01 validation.',
    'required_next_step':'Freeze formula BEFORE evaluating untouched N01 locked test and independent source-excluded holdout.'})



 image_aware__HAM10000 top 10:
cell                                  image_aware__HAM10000
method                                                     
Weighted_SimCLRConcat_0.6                             79.73
vote_SimCLRConcat                                     79.62
arithmetic_SimCLRConcat                               79.62
Weighted_SimCLRConcat_0.4                             79.57
Gate_entropy_SimCLRResidual_beta2                     79.51
geometric_SimCLRConcat                                79.41
Gate_confidence_SimCLRResidual_beta4                  79.40
vote_SimCLRResidual                                   79.40
arithmetic_SimCLRResidual                             79.40
geometric_SimCLRResidual                              79.38

 image_aware__BCN20000 top 10:
cell                             image_aware__BCN20000
method                                                
Base_SimCLR                                      81.54
Weighted_SimCLRResidual_0.8                      81

In [6]:
# Comparisons restricted to identical image IDs across the two protocol-specific source cohorts.
a,b=meta_a,meta_b
shared=sorted(set(a.image_id)&set(b.image_id))
pd.DataFrame({'image_id':shared}).to_csv(OUT/'tables/shared_image_identities.csv',index=False)
paired=[]
if shared:
    ia=pd.Index(a.image_id);ib=pd.Index(b.image_id)
    x=ia.get_indexer(shared);z=ib.get_indexer(shared)
    req((x>=0).all() and (z>=0).all(),'Internal paired ID error')
    req(np.array_equal(a.y.to_numpy()[x],b.y.to_numpy()[z]),'Same ISIC image has contradictory labels across protocols')
    req(np.array_equal(a.source.to_numpy()[x],b.source.to_numpy()[z]),'Same ISIC image has contradictory source metadata')
    for source in ['HAM10000','BCN20000','ISIC2019_pooled_shared']:
        m=np.ones(len(shared),dtype=bool) if source=='ISIC2019_pooled_shared' else a.source.to_numpy()[x]==source
        if m.sum()<2 or len(np.unique(a.y.to_numpy()[x][m]))<2:continue
        for method in RECIPES:
            y=a.y.to_numpy()[x][m]
            pa=PREDICTIONS['image_aware'][method][x][m]
            pb=PREDICTIONS['lesion_aware'][method][z][m]
            sa=metrics(y,pa);sb=metrics(y,pb)
            paired.append({'method':method,'source':source,'n_identical_images':len(y),
                'represented_classes':sa['n_classes'],'image_aware_bacc':sa['balanced_accuracy'],
                'lesion_aware_bacc':sb['balanced_accuracy'],
                'difference_image_minus_lesion_bacc':sa['balanced_accuracy']-sb['balanced_accuracy']})
paired=pd.DataFrame(paired)
paired.to_csv(OUT/'tables/paired_identical_image_comparison.csv',index=False)
print('Verified common image IDs:',len(shared))
if not paired.empty:
    print(paired.loc[paired.method.eq(provisional),['source','n_identical_images','represented_classes','image_aware_bacc','lesion_aware_bacc']].round(4).to_string(index=False))
print('This paired subset is post-hoc and may have poor rare-class representation; it is a diagnostic, not an unbiased head-to-head clinical test.')


Verified common image IDs: 204
                source  n_identical_images  represented_classes  image_aware_bacc  lesion_aware_bacc
              HAM10000                  99                    7            0.8806             0.7202
              BCN20000                 105                    8            0.7398             0.4964
ISIC2019_pooled_shared                 204                    8            0.7592             0.4958
This paired subset is post-hoc and may have poor rare-class representation; it is a diagnostic, not an unbiased head-to-head clinical test.


## Optional full N01 validation replay from original R07 raw probabilities
This cell is optional. It **does not** fill in original J1's unavailable 2,534-image full-test probability archive and **does not** change the four-cell winner. It allows an exact same-catalogue check on the complete 2,186 N01 validation records, instead of G02-B's 2,109 HAM/BCN-only subset.

In [7]:
r07=[]
for p in INPUT.rglob('final_validation.npz'):
    if not p.is_file():continue
    if all((p.parent/f'{s}_meta_val.npz').is_file() for s in ['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']):
        r07.append(p)
if len(r07)==1:
    path=r07[0]
    with np.load(path,allow_pickle=True) as z:
        all_ids=npz_text(z,'image_id');yy=z['y'].astype(int)
        req(npz_text(z,'classes').tolist()==CLASSES,'Optional R07 full validation: class mismatch')
        req(len(all_ids)==2186 and len(set(all_ids))==len(all_ids),'Unexpected full N01 validation IDs')
        rr={}
        for s in SOURCES:
            key={'Rotation':'rotation','SimCLR':'simclr'}.get(s,s)
            with np.load(path.parent/f'{key}_meta_val.npz',allow_pickle=True) as x:
                req(np.array_equal(npz_text(x,'val_image_id'),all_ids) and np.array_equal(x['val_y'].astype(int),yy),f'R07 full validation source mismatch: {key}')
                rr[s]=norm(x['val_proba'])
        optional=[]
        for method,recipe in RECIPES.items():
            st=metrics(yy,ensemble(rr,recipe))
            optional.append({'track':'lesion_aware','cohort':'ISIC2019_N01_full_validation_2186',
                             'method':method,'family':recipe['family'],**st})
        pd.DataFrame(optional).to_csv(OUT/'tables/optional_N01_full_validation_2186_metrics.csv',index=False)
        print('Optional complete N01 validation replay:',len(yy),'images,',len(optional),'methods')
elif r07:
    print('Multiple full R07 prediction archives attached; optional full validation replay skipped, to avoid selecting an arbitrary one.')
else:
    print('R07 full validation output not attached; optional 2,186-image check skipped.')


Optional complete N01 validation replay: 2186 images, 43 methods


## Optional complete original J1 test: all 2,534 images
The historical per-image results notebook saves individual eight-class probabilities under `01_PER_IMAGE_RESULTS/ISIC2019_test/probabilities.npz`. When that exact output is attached, replay the same six-source formula on the full original 2,534-image J1 test. The ISIC full-test scores are **not used to change the source-specific provisional candidate**.

In [8]:
original_npz=[]
for p in INPUT.rglob('probabilities.npz'):
    if p.parent.name=='ISIC2019_test' and '01_PER_IMAGE_RESULTS' in p.parts:original_npz.append(p)
if len(original_npz)==1:
    p=original_npz[0]
    with np.load(p,allow_pickle=True) as z:
        req('labels' in z and 'image_ids' in z and 'class_names' in z,'Original per-image archive has unexpected fields')
        req(npz_text(z,'class_names').tolist()==CLASSES,'Original full J1 test class order differs')
        y=z['labels'].astype(int);ids=npz_text(z,'image_ids')
        req(len(y)==len(ids)==2534 and len(set(ids))==2534,'Original full J1 test identity/count mismatch')
        req(((y>=0)&(y<8)).all(),'Original full J1 test class labels invalid')
        raw={}
        for s in SOURCES:
            k='prob__'+('SimCLR_NoTTA' if s=='SimCLR' else s)
            req(k in z.files,f'Original full J1 archive missing {k}')
            raw[s]=norm(z[k])
        rec=[]
        for method,recipe in RECIPES.items():
            rec.append({'track':'image_aware','cohort':'ISIC2019_J1_full_test_2534',
                        'method':method,'family':recipe['family'],**metrics(y,ensemble(raw,recipe))})
        pd.DataFrame(rec).to_csv(OUT/'tables/optional_original_J1_full_test_2534_metrics.csv',index=False)
        print('Optional complete original J1 test replay:',len(ids),'images,',len(rec),'common methods')
elif original_npz:
    print('Multiple original J1 full-test per-image archives; optional replay skipped to avoid ambiguous source selection.')
else:
    print('Original full J1 per-image results not attached; optional 2,534-image check skipped.')


Original full J1 per-image results not attached; optional 2,534-image check skipped.


In [9]:
# Fixed-formula uncertainty check: bootstrap confidence intervals for the already-observed
# winner versus the unweighted six-model mean. This is not a cure for winner selection bias.
BOOT=int(os.getenv('G04_BOOTSTRAP_N','400'))
req(100<=BOOT<=2000,'G04_BOOTSTRAP_N must be 100..2000')
def bacc(y,p):
    pred=p.argmax(1);return float(np.mean([(pred[y==c]==c).mean() for c in np.unique(y)]))
def boot_delta(y,p1,p0,groups,seed):
    groups=np.asarray(groups,dtype=str)
    groups=np.asarray([g if g.strip() and not g.endswith(':UNKNOWN') else f'UNVERIFIED_IMAGE_{i}' for i,g in enumerate(groups)])
    uniq,inv=np.unique(groups,return_inverse=True)
    indices=[np.flatnonzero(inv==j) for j in range(len(uniq))]
    rng=np.random.default_rng(seed);out=[];refset=set(y)
    for _ in range(BOOT):
        ii=np.concatenate([indices[j] for j in rng.integers(0,len(indices),len(indices))])
        if set(y[ii])!=refset:continue
        out.append(bacc(y[ii],p1[ii])-bacc(y[ii],p0[ii]))
    return {'ci95_low':float(np.percentile(out,2.5)) if len(out)>=50 else None,
            'ci95_high':float(np.percentile(out,97.5)) if len(out)>=50 else None,
            'valid_bootstrap':len(out),'bootstrap_groups':len(uniq)}
bootstrap=[]
for ti,(track,(meta,raw)) in enumerate(TRACKS.items()):
    for ci,source in enumerate(['HAM10000','BCN20000']):
        idx=np.flatnonzero(meta.source.to_numpy()==source);yy=meta.y.to_numpy()[idx]
        p1=PREDICTIONS[track][provisional][idx];p0=PREDICTIONS[track]['arithmetic_All6'][idx]
        res=boot_delta(yy,p1,p0,meta.group_id.to_numpy()[idx],SEED+ti*10+ci)
        bootstrap.append({'track':track,'source':source,'winner':provisional,'baseline':'arithmetic_All6',
            'n':len(idx),'point_delta_bacc':bacc(yy,p1)-bacc(yy,p0),**res})
pd.DataFrame(bootstrap).to_csv(OUT/'tables/winner_vs_mean6_group_bootstrap.csv',index=False)
print(pd.DataFrame(bootstrap).round(4).to_string(index=False))


       track   source                           winner        baseline    n  point_delta_bacc  ci95_low  ci95_high  valid_bootstrap  bootstrap_groups
 image_aware HAM10000 Gate_margin_SimCLRResidual_beta2 arithmetic_All6  999            0.0100   -0.0098     0.0426              400               976
 image_aware BCN20000 Gate_margin_SimCLRResidual_beta2 arithmetic_All6 1240           -0.0007   -0.0278     0.0244              400              1029
lesion_aware HAM10000 Gate_margin_SimCLRResidual_beta2 arithmetic_All6 1060            0.0058   -0.0077     0.0266              397               807
lesion_aware BCN20000 Gate_margin_SimCLRResidual_beta2 arithmetic_All6 1049           -0.0041   -0.0306     0.0214              253               333


In [10]:
summary={'stage':'G04-unified','status':'COMPLETED','seed':SEED,'candidate_count':len(RECIPES),
         'model_source_names':SOURCES,'classes':CLASSES,'g03_sources':[prov_a,prov_b],
         'image_counts':{'image_aware':len(meta_a),'lesion_aware':len(meta_b)},
         'available_populations':['ISIC2019_pooled_source_matched','HAM10000','BCN20000'],
         'shared_image_count':len(shared),'primary_selection_cells':cols,
         'selection_metric':'equal mean BACC across four protocol/source cells (pooled ISIC not counted twice)',
         'provisional_winner':provisional,
         'decision_status':'EXPLORATORY: do not report selected winner as independently validated',
         'model_selection_leakage':'Selection uses previously studied J1 test and N01 validation; cannot claim independent winner confirmation.',
         'domain_independence':'HAM/BCN are subsets of ISIC2019, not target-source-excluded datasets',
         'next_step':'Freeze recipe and confirm on untouched N01 test, then train source-excluded models for true external evaluation',
         'time_seconds':round(time.monotonic()-START,2)}
save_json(OUT/'reports/final_status.json',summary)
print('G04 completed; files:')
for f in sorted(OUT.rglob('*')):
    if f.is_file():print(' ',f.relative_to(OUT),f.stat().st_size,'bytes')


G04 completed; files:
  predictions/image_aware_unified_probabilities.npz 2895132 bytes
  predictions/lesion_aware_unified_probabilities.npz 2735632 bytes
  reports/final_status.json 2802 bytes
  reports/provisional_ensemble_recipe_NOT_FINAL.json 1143 bytes
  reports/unified_frozen_recipes.json 10398 bytes
  tables/all_unified_candidate_metrics.csv 58727 bytes
  tables/optional_N01_full_validation_2186_metrics.csv 10690 bytes
  tables/paired_identical_image_comparison.csv 12997 bytes
  tables/per_class_recall_all_candidates.csv 153392 bytes
  tables/shared_image_identities.csv 2661 bytes
  tables/unified_4cell_exploratory_rankings.csv 12733 bytes
  tables/winner_vs_mean6_group_bootstrap.csv 688 bytes
